# RAG with Qdrant (Chapter 8)

This notebook swaps the in-memory retriever for a persistent Qdrant collection without changing the `SimpleRAG` module. It expects an existing collection whose payload stores text in a `document` field and whose vectors are compatible with `dspy-qdrant`'s configured vectorizer.

**Environment variables**

- `OPENAI_API_KEY`
- `QDRANT_URL`
- `QDRANT_API_KEY` (optional for an unauthenticated local server)

Set `QDRANT_URL=http://127.0.0.1:6333` for a local Qdrant server.


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
%pip install dspy-qdrant qdrant-client fastembed -q


In [ ]:
import os

import dspy
from dotenv import load_dotenv
from dspy_qdrant import QdrantRM
from qdrant_client import QdrantClient

load_dotenv()
dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION = os.getenv("QDRANT_COLLECTION", "knowledge_base")


## Connect the retriever


In [ ]:
if QDRANT_URL:
    client = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
    )
    search = QdrantRM(
        qdrant_collection_name=COLLECTION,
        qdrant_client=client,
        document_field="document",
        k=5,
    )
    print(f"Connected QdrantRM to {COLLECTION!r} at {QDRANT_URL}.")
else:
    search = None
    print("Set QDRANT_URL before running the retrieval example.")


## Reuse the fixed RAG module


In [ ]:
class SimpleRAG(dspy.Module):
    def __init__(self, retriever):
        super().__init__()
        self.retriever = retriever
        self.respond = dspy.ChainOfThought(
            "context, question -> response"
        )

    def forward(self, question):
        context = self.retriever(question).passages
        return self.respond(
            context=context,
            question=question,
        )


if search is not None:
    rag = SimpleRAG(retriever=search)
    result = rag(question="How do I optimize a DSPy program?")
    print(result.response)
else:
    print("Skipping RAG call because QDRANT_URL is not configured.")


Because `search` has the same `.passages` return interface as the in-memory retriever, it can also replace that retriever inside the chapter's `search_knowledge_base` ReAct tool. In production, make sure the collection's vector dimensions, vector name, embedding model, and `document_field` match the `QdrantRM` configuration.
